# Software profesional en Acústica 2025-26 (M2i)

*This notebook contains a modification of the notebook [FEM_Helmholtz_equation_Robin](https://github.com/spatialaudio/computational_acoustics/blob/master/FEM_Helmholtz_equation_Robin.ipynb), created by Sascha Spors, Frank Schultz, Computational Acoustics Examples, 2018. The text/images are licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/). The FEniCS implementation has been replaced with NGSolve. The code is released under the [MIT license](https://opensource.org/licenses/MIT).*

First, we need to install on the fly [NGSolve](https://ngsolve.org/) using the [FEM on Colab](https://fem-on-colab.github.io/packages.html) install script:

In [ ]:
try:
    import ngsolve
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/ngsolve-install-release-complex.sh" -O "/tmp/ngsolve-install.sh" && bash "/tmp/ngsolve-install.sh"
    import ngsolve

# Numerical Solution of the Helmholtz Equation in unbounded domains with Perfectly Matched Layers and a Finite Element Method

This notebook illustrates the numerical solution of the wave equation for an harmonic excitation stated in an unbounded domain using a Finite Element Method. To truncate the unbounded domain, the Perfectly Matched Layer (PML) technique is used. 

## Problem Statement

The homogeneous Helmholtz equation is given as

\begin{equation}
-c^2\Delta P(\boldsymbol{x}) - \omega^2 P(\boldsymbol{x}) = 0\qquad \text{for }\quad\boldsymbol{x}=(x,y)\in\Omega_F
\end{equation}

where $\Omega_F$ is the fluid subdomain which where the numerical solution is computed. This subdomain $\Omega_F$ is surrounded by an artificial sponge layer, where the PML goberning equations are stated:

\begin{equation}
-c^2\mathrm{div}(\mathcal{C}\nabla P(\boldsymbol{x}) - \omega^2 \mathcal{M} P(\boldsymbol{x}) = 0\qquad \text{for }\quad\boldsymbol{x}=(x,y)\in\Omega_{PML}
\end{equation}

where
$$
\mathcal{C}=
\begin{pmatrix}
\frac{\gamma_y}{\gamma_x} & 0\\
0 & \frac{\gamma_x}{\gamma_y}
\end{pmatrix},
\qquad
\mathcal{M} = \gamma_x\gamma_y
$$

with
$$
\gamma_{x}(x)=1+\frac{i}{\omega}\sigma_{x}(s),\qquad \gamma_{y}(y)=1+\frac{i}{\omega}\sigma_{y}(y)
$$

The function $\sigma_x$ and $\sigma_y$ are the so-called PML absorption profiles. These functions can be constant, quadratic or even singular. The only requirements are: $\sigma_x$ and $\sigma_y$ are positive and monotonically increasing.

The set of governing equations must be completed with boundary conditions in the fluid subdomain (in this case, Dirichlet boundary conditions will be used $P=1$ in the interior fluid boundary), and also null Dirichlet boundary conditions will be used in the exterior boundary of the PML subdomain $P=0$. Finally, to couple both fluid and PML subdomains, it is imposed
\begin{align}
u|_{\Omega_{F}}=u|_{\Omega_{PML}}\qquad \text{on }\quad\partial\Omega_{F}\cap \partial\Omega_{PML},\\
\nabla u|_{\Omega_{F}}\cdot \boldsymbol{n}=\mathcal{C}\nabla u|_{\Omega_{PML}}\cdot \boldsymbol{n}\qquad \text{on }\quad\partial\Omega_{F}\cap \partial\Omega_{PML},\\
\end{align}

which are the natural coupling boundary conditions to avoid any reflection between both subdomains.

## Variational Formulation

If we extend the definition of the absorption PML profiles in the fluid subdomain as $\sigma_x(x)=0$ and $\sigma_y(y)=0$ for $\boldsymbol{x}=(x,y)\in\Omega_{F}$, then the Helmholtz equation and the PML equation can be written in the same manner in $\Omega=\Omega_F\cup\Omega_{PML}$. Again using a Green's formula, it holds that the pressure field $P$

\begin{equation*}
c^2\int_\Omega \mathcal{C}\nabla P\cdot \nabla Q \,\mathrm{d}\boldsymbol{x} -\omega^2\int_{\Omega}\mathcal{M} PQ\,\mathrm{d}\boldsymbol{x}=\int_\Omega FQ \,\mathrm{d}\boldsymbol{x},
\end{equation*}

for all test functions $Q$
It is common to express this integral equation in terms of the bilinear $a(P, Q)$ and linear $L(Q)$ forms 

\begin{equation*}
a(P,Q) = c^2\int_\Omega \mathcal{C}\nabla P\cdot \nabla Q \,\mathrm{d}\boldsymbol{x} -\omega^2\int_{\Omega}\mathcal{M} PQ\,\mathrm{d}\boldsymbol{x},
\end{equation*}
and
\begin{equation*}
L(Q) = \int_\Omega FQ \,\mathrm{d}\boldsymbol{x},
\end{equation*}

where

\begin{equation*}
a(P, Q) = L(Q)\qquad \forall Q.
\end{equation*}

In this case, the source term $F=0$.



## Numerical Solution

The numerical solution of the variational problem is based on [NGSolve](https://ngsolve.org/), an open-source framework for numerical solution of PDEs.
Its high-level Python interface is used in the following to define the problem and compute the solution.
The implementation is based on the variational formulation derived above.
The definition of the problem in NGSolve is very close to the mathematical formulation of the problem.

#### Import modules and define the physical data and the geometrical setting

In [ ]:
import numpy as np
import scipy.special as spe
import matplotlib.pyplot as plt
from netgen.geom2d import SplineGeometry
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw

# Parameter values
amplitude = 1.0  # amplitude [Pa]
degree = 0       # number of the Fourier mode in the exact solution
omega = 2*np.pi*200.  # angular frequency [rad/s]
vel = 340        # sound speed [m/s]

# Geometrical setting
Radius = 1.0          # radius of a circular obstacle centered at (0,0)
Lx = 2.0; Ly = 2.0   # dimensions of the fluid computational domain
th = 0.75 * vel / (omega / (2.*np.pi))  # PML thickness = 1.5*wavelength

#### Compute mesh

In [ ]:
# Build geometry using Netgen
domain_exterior = MoveTo(-Lx, -Ly).Rectangle(2*Lx, 2*Ly).Face()
domain_interior = MoveTo(-Lx-th, -Ly-th).Rectangle(2*(Lx+th), 2*(Ly+th)).Face()
obstacle = inner = Circle((0,0), Radius).Face()

domain_PML = domain_exterior- domain_interior
domain_fluid = domain_interior - obstacle

# Domain and boundary tags for the PML domain
domain_PML.faces.name = "pml"
domain_PML.faces.col = (0, 1, 0)  # Green 
domain_PML.mat("pml")
domain_PML.edges.Min(X).name = "outer"
domain_PML.edges.Min(X).col = (1, 0, 0) # red
domain_PML.edges.Max(X).name = "outer"
domain_PML.edges.Max(X).col = (1, 0, 0) # red
domain_PML.edges.Min(Y).name = "outer"
domain_PML.edges.Min(Y).col = (1, 0, 0) # red
domain_PML.edges.Max(Y).name = "outer"
domain_PML.edges.Max(Y).col = (1, 0, 0) # red

# Domain and boundary tags for the fluid domain
domain_fluid.faces.name = "fluid"
domain_fluid.faces.col = (0, 0, 1)  # Blue 
domain_fluid.mat("fluid")
domain_fluid.edges.name = "inner"
domain_fluid.edges.col = (0, 1, 0) # red
domain_fluid.edges.Min(Y).name = "interface"
domain_fluid.edges.Min(Y).col = (1, 0, 1) 
domain_fluid.edges.Min(X).name = "interface"
domain_fluid.edges.Min(X).col = (1, 0, 1) 
domain_fluid.edges.Max(X).name = "interface"
domain_fluid.edges.Max(X).col = (1, 0, 1) 
domain_fluid.edges.Max(X).name = "interface"
domain_fluid.edges.Max(X).col = (1, 0, 1) 

# Generate geomerey
geo = OCCGeometry(Glue([domain_fluid , domain_PML]), dim=2)

# Generate mesh
ngmesh = geo.GenerateMesh(maxh=0.1)
mesh = Mesh(ngmesh)
mesh.Curve(3)  # Curve order 3 for better approximation of the circle

Draw(mesh, height="3vh")
print("Number of vertices:", mesh.nv)

#### Define the functional spaces and the integral measures

In [ ]:
# Define function space: mixed H1 x H1 for real and imaginary parts
V = H1(mesh, order=1, dirichlet="outer|inner", complex=True)

# Define variational unknowns and test functions
u, v = fes.TnT()

#### Define source terms and Dirichlet boundary data

In [ ]:
# Boundary data on the circle: P = exp(i*n*theta)
n_mode = degree
g = np.exp(1j * n_mode * atan2(y, x))

# Apply boundary conditions
gfu = GridFunction(fes)

# BCs: on circle_boundary -> (g_re, g_im), on pml_boundary -> (0, 0)
gfu.Set(g, definedon=mesh.Boundaries("inner"))

# pml_boundary is already 0 by default (homogeneous Dirichlet)
gfu.Set(CF(0.), definedon=mesh.Boundaries("outer"))

#### Define the fluid and the PML coefficients

In [ ]:
# PML absorption profile (quadratic)
sigma0 = 2e3

# sx, sy as NGSolve CoefficientFunctions
s_x = IfPos(Abs(x) - Lx, sigma0 * (Abs(x) - Lx)**2 / omega, 0.0)
s_y = IfPos(Abs(y) - Ly, sigma0 * (Abs(y) - Ly)**2 / omega, 0.0)

# PML coefficients (real and imaginary parts of gamma_x/gamma_y and gamma_x*gamma_y)
gamma_x= 1 + 1j * sigma_x / omega
gamma_y= 1 + 1j * sigma_y / omega

# PML tensor components (2x2 diagonal matrices as CoefficientFunctions)
C = CoefficientFunction((gamma_y/gamma_x, 0, 0, gamma_x/gamma_y), dims=(2,2))
M = gamma_x * gamma_y

#### Define the variational problem and compute the FEM solution

In [ ]:
# Physical constants
c2 = vel**2
w2 = omega**2

# Bilinear form
a = BilinearForm(fes)
a += c2 * InnerProduct(C * grad(u) , grad(v)) * dx
a += -w2 * M * u * v * dx

# Linear form (zero source)
f = LinearForm(fes)
f += 0 * v * dx

# Assemble
a.Assemble()
f.Assemble()

# Apply Dirichlet boundary conditions (explicitly) and solve
r = f.vec.CreateVector()
r.data = f.vec - a.mat * gfu.vec
gfu.vec.data += a.mat.Inverse(fes.FreeDofs()) * r

In [ ]:
# Plot the real part and the magnitude of the pressure field
Draw(gfu.real**2 + gfu.imag**2, mesh, "|P|^2", height="3vh")

# Complex-valñued solution (animated)
Draw(gfu, mesh, animate_complex=True,, height="3vh")

#### Define the exact solution and compute the $L^2$- relative error in the fluid subdomain

In this case, since the Dirichlet data is an harmonic expression (this is, a complex-exponential $e^{in\theta}$) on the interior circle, the solution is given by
$$
P(r,\theta)=e^{in\theta}H^{(1)}_{n}(kr)
$$
with $k=\omega/c$, $r=\sqrt{x^2+y^2}$, and $\theta=\mathrm{arctan}(y/x)$.

In [ ]:
# Physical parameters
k = omega / vel
norm_factor = spe.hankel1(degree, k * Radius)

# Exact solution
uex = CF(np.exp(1j*degree*np.atan2(x, y)) * spe.hankel1(degree, k*sqrt(x**2+y**2)) / norm_factor)

# Computing relative error
diff = u - uex
err  = sqrt(Integrate(InnerProduct(diff, diff), mesh, definedon=mesh.Materials("fluid")))
norm = sqrt(Integrate(InnerProduct(uex_cf, uex_cf), mesh, definedon=mesh.Materials("fluid")))
print("L2-relative error (%): ", err / norm * 100.)

**Copyright**

This notebook is provided as [Open Educational Resource](https://en.wikipedia.org/wiki/Open_educational_resources). Feel free to use the notebook for your own purposes. The text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/), the code of the IPython examples under the [MIT license](https://opensource.org/licenses/MIT).